# 🧊 GloVe — Global Vectors for Word Representation

> **Goal:** Understand how GloVe works, how it differs from Word2Vec, how to build a co-occurrence matrix from scratch, and how to use pretrained GloVe embeddings.

---

## 📖 Table of Contents
1. [The Core Idea — What's Different from Word2Vec?](#1)
2. [Setup & Imports](#2)
3. [The Co-occurrence Matrix — GloVe's Foundation](#3)
4. [GloVe's Objective Function (Intuition)](#4)
5. [Building a Toy Co-occurrence Matrix from Scratch](#5)
6. [Training GloVe from Scratch (Simplified)](#6)
7. [Using Pretrained GloVe Embeddings](#7)
8. [Vector Arithmetic with GloVe](#8)
9. [Visualizing GloVe Embeddings](#9)
10. [Word2Vec vs GloVe — Head-to-Head Comparison](#10)

---

<a id='1'></a>
## 1. 🎯 The Core Idea — What's Different from Word2Vec?

### Word2Vec's Approach (Local)
Word2Vec slides a **small window** across the text, one step at a time, learning from local context pairs.

> Like a person reading a book one sentence at a time, never stepping back to see the big picture.

### GloVe's Approach (Global)
GloVe first **counts every co-occurrence** across the entire corpus, building a global statistics table. Then it factorizes that table into word vectors.

> Like a librarian who reads the ENTIRE library first, builds a comprehensive index of which words appear near which, and *then* derives meaning from the statistics.

---

### 📚 The Library Analogy

| Method | Analogy |
|---|---|
| **Word2Vec** | A student reading one page at a time, guessing the next word |
| **GloVe** | A librarian who builds a full cross-reference index, then finds patterns in the stats |

---

### The GloVe Insight

GloVe was published by Stanford in 2014 (Pennington, Socher, Manning). The key insight is:

> **Ratios of co-occurrence probabilities** encode meaning, not just raw counts.

```
P('ice' | 'solid')   is HIGH    (ice and solid co-occur a lot)
P('steam' | 'solid') is LOW     (steam and solid don't co-occur much)

Ratio: P(solid|ice) / P(solid|steam)  ≫  1
```

This ratio tells us: *ice is more related to solid than steam is*. GloVe learns vectors that preserve these ratios.

<a id='2'></a>
## 2. 📦 Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
import warnings, os, re
from collections import defaultdict, Counter
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from numpy.linalg import norm
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        12,
})

print("✅ Libraries imported!")
print("\n📌 GloVe overview:")
print("   Paper  : 'GloVe: Global Vectors for Word Representation'")
print("   Authors: Pennington, Socher, Manning (Stanford, 2014)")
print("   Key    : Factorize a global co-occurrence matrix")

<a id='3'></a>
## 3. 📊 The Co-occurrence Matrix — GloVe's Foundation

A **co-occurrence matrix** $X$ has:
- Rows = words
- Columns = context words  
- $X_{ij}$ = how many times word $j$ appears in the context of word $i$

### 🗓️ Timetable Analogy

> Imagine a school timetable. Instead of students and subjects, we have words. The cell at row `king`, column `queen` tells you: how often did 'queen' appear near 'king' in the entire corpus?
>
> Words that frequently share contexts get large values. Unrelated words get zeros or near-zeros.

In [ ]:
# ── Build the co-occurrence matrix ────────────────────────────────
corpus = [
    "the king ruled the kingdom with great wisdom",
    "the queen ruled beside the king with grace",
    "the prince is the son of the king",
    "the princess is the daughter of the queen",
    "a man worked hard every day to earn gold",
    "a woman worked hard to raise her children",
    "the king and queen lived in a royal palace",
    "the royal family celebrated in the palace",
    "a dog barked loudly at the cat by the fence",
    "the cat chased the small mouse in the garden",
    "ice is a solid form of water and feels cold",
    "steam is a gas form of water and feels hot",
    "water can be solid ice or hot steam",
    "the doctor treated the sick patient carefully",
    "the nurse helped the doctor in the hospital",
    "the teacher taught the student at school",
    "paris is the capital city of france",
    "berlin is the capital city of germany",
    "france and germany are european countries",
    "the scientist worked in the laboratory all day",
]

def build_cooccurrence(corpus, window=2):
    """
    Build a co-occurrence matrix from a corpus.
    Returns: matrix, word2idx, idx2word
    """
    # Tokenize
    all_words = []
    tokenized = []
    for sent in corpus:
        tokens = re.findall(r'\w+', sent.lower())
        tokenized.append(tokens)
        all_words.extend(tokens)

    # Build vocabulary (words appearing >= 2 times)
    freq = Counter(all_words)
    vocab = sorted([w for w, c in freq.items() if c >= 2])
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    V = len(vocab)

    # Count co-occurrences
    X = np.zeros((V, V))
    for tokens in tokenized:
        for i, center in enumerate(tokens):
            if center not in word2idx:
                continue
            ci = word2idx[center]
            for j in range(max(0, i-window), min(len(tokens), i+window+1)):
                if i == j or tokens[j] not in word2idx:
                    continue
                cj = word2idx[tokens[j]]
                # Distance-weighted: closer = more weight
                weight = 1.0 / abs(i - j)
                X[ci, cj] += weight

    return X, word2idx, idx2word, vocab

X, word2idx, idx2word, vocab = build_cooccurrence(corpus, window=2)
V = len(vocab)

print(f"Co-occurrence matrix built:")
print(f"  Vocabulary size: {V}")
print(f"  Matrix shape   : {X.shape}")
print(f"  Non-zero cells : {np.count_nonzero(X)}")
print(f"  Sparsity       : {1 - np.count_nonzero(X)/(V*V):.2%}")
print(f"\nSample co-occurrence counts (king, queen, man, woman, dog, cat):")
sample = ['king', 'queen', 'man', 'woman', 'dog', 'cat']
sample = [w for w in sample if w in word2idx]
print(f"{'':12}", end="")
for w in sample:
    print(f"{w:10}", end="")
print()
for w1 in sample:
    print(f"{w1:12}", end="")
    for w2 in sample:
        print(f"{X[word2idx[w1], word2idx[w2]]:10.2f}", end="")
    print()

In [ ]:
# ── Visualize the co-occurrence matrix ────────────────────────────
focus_words = ['king', 'queen', 'man', 'woman', 'prince', 'princess',
               'dog', 'cat', 'doctor', 'nurse', 'ice', 'steam', 'water']
focus_words = [w for w in focus_words if w in word2idx]

idx_focus = [word2idx[w] for w in focus_words]
X_focus   = X[np.ix_(idx_focus, idx_focus)]

fig, ax = plt.subplots(figsize=(11, 9))

# Use log scale for better visualization
X_plot = X_focus + 0.01  # avoid log(0)
im = ax.imshow(X_plot, cmap='YlOrRd', norm=LogNorm(vmin=0.01, vmax=X_plot.max()))
plt.colorbar(im, ax=ax, label='Co-occurrence count (log scale)')

ax.set_xticks(range(len(focus_words)))
ax.set_yticks(range(len(focus_words)))
ax.set_xticklabels(focus_words, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(focus_words, fontsize=11)

for i in range(len(focus_words)):
    for j in range(len(focus_words)):
        val = X_focus[i, j]
        if val > 0:
            ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                    fontsize=8, color='black' if val < 2 else 'white')

ax.set_title('Co-occurrence Matrix (window=2)\nBrighter = more frequent co-occurrence',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cooccurrence_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print("💡 Notice:")
print("  • king & queen have high co-occurrence → they'll have similar vectors")
print("  • dog & cat co-occur → they'll cluster together")
print("  • ice & water co-occur → related vectors")
print("  • king & dog rarely co-occur → distant vectors")

<a id='4'></a>
## 4. 📐 GloVe's Objective Function (Intuition)

GloVe's goal is to learn word vectors $w_i$ and $w_j$ such that:

$$w_i^T w_j + b_i + b_j \approx \log(X_{ij})$$

The **dot product** of two word vectors should equal the **log of how often they co-occur**.

### 🌡️ Thermometer Analogy
> Imagine two thermometers: one for each word. If two words frequently appear near each other, their thermometers should read similar temperatures (high dot product). If they rarely co-occur, their thermometers should point in very different directions (low dot product).

### The Weighting Function

Not all co-occurrences are equally important. GloVe uses a weighting function:

$$f(X_{ij}) = \begin{cases} (X_{ij}/x_{max})^\alpha & \text{if } X_{ij} < x_{max} \\ 1 & \text{otherwise} \end{cases}$$

This **down-weights very frequent pairs** (like `the-a`) and **up-weights rare but informative pairs**.

In [ ]:
# ── Visualize the GloVe weighting function ────────────────────────
def f_weight(x, x_max=100, alpha=0.75):
    """GloVe weighting function"""
    return np.where(x < x_max, (x / x_max) ** alpha, 1.0)

x_vals = np.linspace(0, 200, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Weighting function
ax = axes[0]
for alpha, color, label in [(0.5, '#4A90D9', 'α=0.5'),
                              (0.75, '#F5A623', 'α=0.75 (default)'),
                              (1.0, '#E87070', 'α=1.0')]:
    ax.plot(x_vals, f_weight(x_vals, alpha=alpha), color=color,
            linewidth=2.5, label=label)

ax.axvline(100, color='#888', linestyle='--', alpha=0.6, label='x_max=100')
ax.axhline(1.0, color='#888', linestyle=':', alpha=0.4)
ax.set_xlabel('Co-occurrence count $X_{ij}$', fontsize=12)
ax.set_ylabel('Weight $f(X_{ij})$', fontsize=12)
ax.set_title('GloVe Weighting Function\n(caps the influence of very frequent pairs)',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

ax.annotate('Rare pairs:\nget low weight', xy=(10, 0.1), fontsize=10,
            color='#555', style='italic')
ax.annotate('Very frequent pairs:\ncapped at 1.0', xy=(110, 0.9), fontsize=10,
            color='#555', style='italic')

# Right: Log co-occurrence (GloVe target)
ax2 = axes[1]
xij_vals = np.linspace(0.1, 50, 300)
ax2.plot(xij_vals, np.log(xij_vals), color='#7B68EE', linewidth=2.5)
ax2.fill_between(xij_vals, np.log(xij_vals), alpha=0.15, color='#7B68EE')
ax2.set_xlabel('Co-occurrence count $X_{ij}$', fontsize=12)
ax2.set_ylabel('$\\log(X_{ij})$', fontsize=12)
ax2.set_title('GloVe Target: log(co-occurrence)\n$w_i^T w_j \\approx \\log(X_{ij})$',
             fontweight='bold')
ax2.grid(True, alpha=0.3)

# Annotate some points
for xval, label in [(1, 'rarely\nco-occur'), (10, 'sometimes'), (40, 'frequently\nco-occur')]:
    ax2.scatter([xval], [np.log(xval)], s=80, zorder=5, color='#F5A623')
    ax2.annotate(f'X={xval}\nlog={np.log(xval):.1f}', xy=(xval, np.log(xval)),
                xytext=(xval+2, np.log(xval)-0.5), fontsize=9, color='#555')

plt.tight_layout()
plt.show()

print("💡 Key insight: GloVe doesn't predict words — it learns vectors whose")
print("   dot products match the LOG of the co-occurrence counts.")

<a id='5'></a>
## 5. 🔧 Building a Toy Co-occurrence Matrix from Scratch

Let's trace through exactly what GloVe sees when it processes a sentence.

In [ ]:
# ── Step-by-step co-occurrence counting ──────────────────────────
def trace_cooccurrence(sentence, window=2):
    words = sentence.lower().split()
    counts = defaultdict(float)
    
    print(f"Sentence: \"{sentence}\"")
    print(f"Window size: {window}")
    print()
    print(f"{'Center':<12} {'Context':<12} {'Distance':<10} {'Weight':<10} {'Co-occurrence pair'}")
    print("-" * 65)
    
    for i, center in enumerate(words):
        for j in range(max(0, i-window), min(len(words), i+window+1)):
            if i == j:
                continue
            ctx  = words[j]
            dist = abs(i - j)
            w    = 1.0 / dist
            pair = tuple(sorted([center, ctx]))
            counts[pair] += w
            print(f"  {center:<12} {ctx:<12} {dist:<10} {w:.2f}       {center} ↔ {ctx}")
    
    print()
    print("Aggregated symmetric co-occurrence counts:")
    print("-" * 40)
    for pair, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {pair[0]:12} ↔ {pair[1]:12}  count: {count:.2f}")

trace_cooccurrence("the king ruled the kingdom", window=2)

In [ ]:
# ── The co-occurrence ratio insight ──────────────────────────────
print("🔑 THE KEY INSIGHT: Co-occurrence RATIOS encode meaning\n")
print("Given a probe word k, and two target words i and j:")
print("   ratio = P(k | i) / P(k | j)\n")

# Let's compute P(context | word) from our matrix
# P(k | i) = X[i,k] / sum(X[i,:])
row_sums = X.sum(axis=1, keepdims=True) + 1e-9
P = X / row_sums   # conditional probability matrix

examples = [
    ('ice', 'steam', 'king',    "king (irrelevant probe)"),
    ('ice', 'steam', 'water',   "water (relevant to both)"),
    ('king', 'queen', 'royal',  "royal (relevant probe)"),
    ('king', 'queen', 'dog',    "dog (irrelevant probe)"),
]

print(f"{'Word i':<10} {'Word j':<10} {'Probe k':<12} {'P(k|i)':<12} {'P(k|j)':<12} {'Ratio':<10} {'Meaning'}")
print("-" * 90)

for wi, wj, wk, meaning in examples:
    if not all(w in word2idx for w in [wi, wj, wk]):
        continue
    pi = P[word2idx[wi], word2idx[wk]]
    pj = P[word2idx[wj], word2idx[wk]]
    ratio = (pi + 1e-9) / (pj + 1e-9)
    print(f"{wi:<10} {wj:<10} {wk:<12} {pi:<12.4f} {pj:<12.4f} {ratio:<10.2f} {meaning}")

print()
print("💡 Interpretation:")
print("  ratio >> 1 → probe is more associated with word i")
print("  ratio ≈ 1  → probe is equally associated with both")
print("  ratio << 1 → probe is more associated with word j")
print("\nGloVe learns vectors that CAPTURE these ratio patterns!")

<a id='6'></a>
## 6. 🏋️ Training GloVe from Scratch (Simplified)

GloVe minimizes the following loss:

$$J = \sum_{i,j=1}^{V} f(X_{ij}) \left( w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

Where:
- $w_i$ = word vector for word $i$
- $\tilde{w}_j$ = context vector for word $j$  
- $b_i, \tilde{b}_j$ = bias terms
- $f(X_{ij})$ = weighting function
- Final vector = $w + \tilde{w}$ (sum of both vectors)

In [ ]:
# ── GloVe training from scratch ───────────────────────────────────
def f_weight(x, x_max=5, alpha=0.75):
    return np.where(x < x_max, (x / x_max) ** alpha, 1.0)

def train_glove(X, V, N=20, lr=0.05, epochs=300, x_max=5, alpha=0.75):
    """
    Simplified GloVe training using AdaGrad.
    Returns: word vectors (W + W_ctx), loss history
    """
    np.random.seed(42)
    # Word vectors and context vectors
    W     = (np.random.rand(V, N) - 0.5) / N
    W_ctx = (np.random.rand(V, N) - 0.5) / N
    b     = np.zeros(V)
    b_ctx = np.zeros(V)

    # AdaGrad accumulators
    grad_sq_W     = np.ones_like(W)
    grad_sq_Wctx  = np.ones_like(W_ctx)
    grad_sq_b     = np.ones(V)
    grad_sq_bctx  = np.ones(V)

    losses = []

    # Pre-compute log X (only where X > 0)
    log_X = np.zeros_like(X)
    mask  = X > 0
    log_X[mask] = np.log(X[mask])

    # Pre-compute weights
    weights = f_weight(X, x_max=x_max, alpha=alpha)

    for epoch in range(epochs):
        total_loss = 0.0

        for i in range(V):
            for j in range(V):
                if X[i, j] == 0:
                    continue

                # Inner difference: dot product should equal log co-occurrence
                diff = W[i] @ W_ctx[j] + b[i] + b_ctx[j] - log_X[i, j]
                w    = weights[i, j]

                total_loss += 0.5 * w * diff ** 2

                # Gradients
                grad_wi    = w * diff * W_ctx[j]
                grad_wj    = w * diff * W[i]
                grad_bi    = w * diff
                grad_bj    = w * diff

                # AdaGrad update
                grad_sq_W[i]    += grad_wi ** 2
                grad_sq_Wctx[j] += grad_wj ** 2
                grad_sq_b[i]    += grad_bi ** 2
                grad_sq_bctx[j] += grad_bj ** 2

                W[i]     -= lr * grad_wi    / np.sqrt(grad_sq_W[i])
                W_ctx[j] -= lr * grad_wj    / np.sqrt(grad_sq_Wctx[j])
                b[i]     -= lr * grad_bi    / np.sqrt(grad_sq_b[i])
                b_ctx[j] -= lr * grad_bj    / np.sqrt(grad_sq_bctx[j])

        losses.append(total_loss)
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1:4d}/{epochs} | Loss: {total_loss:.4f}")

    # GloVe convention: final vector = sum of word + context vector
    vectors = W + W_ctx
    return vectors, losses

print("🚀 Training GloVe from scratch...")
print("   (This may take ~30 seconds for a clean implementation)")
glove_vectors, glove_losses = train_glove(X, V, N=20, epochs=300)
print("\n✅ GloVe training complete!")
print(f"   Vector shape: {glove_vectors.shape}  ({V} words × 20 dims)")

In [ ]:
# ── Plot training loss ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(glove_losses, color='#26A69A', linewidth=2.5)
ax.fill_between(range(len(glove_losses)), glove_losses, alpha=0.15, color='#26A69A')
ax.set_xlabel('Epoch')
ax.set_ylabel('GloVe Loss')
ax.set_title('GloVe Training Loss\n(Fitting dot products to log co-occurrence counts)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check similarity from our scratch model
def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2) + 1e-9)

def top_similar_glove(word, vectors, word2idx, idx2word, top_n=5):
    if word not in word2idx:
        print(f"'{word}' not in vocab"); return
    wi = word2idx[word]
    sims = [(idx2word[i], cosine_sim(vectors[wi], vectors[i]))
            for i in range(len(idx2word)) if i != wi]
    return sorted(sims, key=lambda x: -x[1])[:top_n]

print("\n📊 Similarity from scratch GloVe:")
for query in ['king', 'water', 'doctor']:
    if query in word2idx:
        sims = top_similar_glove(query, glove_vectors, word2idx, idx2word)
        print(f"\n  '{query}' → ", [(w, f"{s:.3f}") for w, s in sims])

<a id='7'></a>
## 7. 🌍 Using Pretrained GloVe Embeddings

Stanford provides pretrained GloVe vectors trained on billions of words. We'll simulate using them here by creating a GloVe-style embedding loader that works with downloaded files.

**Official download:** https://nlp.stanford.edu/projects/glove/

Available pretrained models:
- `glove.6B.50d.txt`  — 6B tokens, 50 dims (smallest, fastest)
- `glove.6B.100d.txt` — 6B tokens, 100 dims
- `glove.6B.300d.txt` — 6B tokens, 300 dims (best quality)
- `glove.840B.300d.txt` — 840B tokens, 300 dims (largest)

In [ ]:
# ── GloVe loader function (works with downloaded .txt files) ─────

def load_glove_embeddings(filepath):
    """
    Load GloVe embeddings from a .txt file.
    
    File format (one word per line):
        word  0.418  0.24968  -0.41242  ...\n
    
    Usage:
        embeddings = load_glove_embeddings('glove.6B.50d.txt')
        king_vec = embeddings['king']  # numpy array of shape (50,)
    """
    embeddings = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            embeddings[word] = vec
    print(f"  Loaded {len(embeddings):,} word vectors")
    print(f"  Embedding size: {len(next(iter(embeddings.values())))}")
    return embeddings

def most_similar_glove(word, embeddings, topn=10):
    """
    Find most similar words using cosine similarity.
    """
    if word not in embeddings:
        print(f"'{word}' not in GloVe vocabulary")
        return []
    
    query_vec = embeddings[word]
    # Vectorized similarity across all words
    words  = list(embeddings.keys())
    matrix = np.array(list(embeddings.values()))
    
    norms  = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9
    normed = matrix / norms
    q_norm = query_vec / (np.linalg.norm(query_vec) + 1e-9)
    
    sims   = normed @ q_norm
    top_idx = np.argsort(sims)[::-1][1:topn+1]
    
    return [(words[i], sims[i]) for i in top_idx]

def glove_analogy(embeddings, positive, negative, topn=5):
    """
    Compute: result ≈ sum(positive) - sum(negative)
    e.g., glove_analogy(emb, ['king','woman'], ['man']) → 'queen'
    """
    query = sum(embeddings[w] for w in positive if w in embeddings)
    query -= sum(embeddings[w] for w in negative if w in embeddings)
    
    exclude = set(positive + negative)
    words   = [w for w in embeddings if w not in exclude]
    matrix  = np.array([embeddings[w] for w in words])
    
    norms  = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9
    normed = matrix / norms
    q_norm = query / (np.linalg.norm(query) + 1e-9)
    sims   = normed @ q_norm
    
    top_idx = np.argsort(sims)[::-1][:topn]
    return [(words[i], sims[i]) for i in top_idx]

print("✅ GloVe loader functions ready!")
print()
print("To use with real GloVe embeddings:")
print("  1. Download from: https://nlp.stanford.edu/projects/glove/")
print("  2. Run: embeddings = load_glove_embeddings('glove.6B.50d.txt')")
print("  3. Use: most_similar_glove('king', embeddings)")
print()
print("─" * 60)
print("We'll now simulate GloVe using our trained toy vectors.")

In [ ]:
# ── Simulate using our GloVe vectors like pretrained ones ─────────
# Build the same API as the real GloVe loader
toy_glove_embeddings = {idx2word[i]: glove_vectors[i] for i in range(V)}

print("📊 GloVe Similarity Results (from toy corpus):")
print("=" * 50)

for query_word in ['king', 'water', 'doctor', 'cat']:
    if query_word in toy_glove_embeddings:
        results = most_similar_glove(query_word, toy_glove_embeddings, topn=5)
        print(f"\n  Most similar to '{query_word}':")
        for word, sim in results:
            bar = '█' * max(0, int(sim * 25))
            print(f"    {word:<15} {sim:+.4f}  {bar}")

<a id='8'></a>
## 8. ➕➖ Vector Arithmetic with GloVe

GloVe is particularly known for excellent performance on **analogy tasks** because its global co-occurrence statistics capture robust relational patterns.

In [ ]:
# ── Analogy tests ─────────────────────────────────────────────────
print("🔮 GloVe Vector Analogies:")
print("=" * 60)

analogy_tests = [
    (['king', 'woman'], ['man'],    "king − man + woman ≈ ?"),
    (['queen', 'man'],  ['woman'],  "queen − woman + man ≈ ?"),
    (['doctor', 'woman'], ['man'],  "doctor − man + woman ≈ ?"),
    (['paris', 'germany'], ['france'], "paris − france + germany ≈ ?"),
    (['prince', 'woman'], ['man'],  "prince − man + woman ≈ ?"),
]

for pos, neg, desc in analogy_tests:
    if not all(w in toy_glove_embeddings for w in pos + neg):
        print(f"\n  {desc}")
        print(f"    (some words not in vocabulary)")
        continue
    results = glove_analogy(toy_glove_embeddings, pos, neg, topn=3)
    print(f"\n  {desc}")
    for word, sim in results:
        bar = '█' * max(0, int(sim * 25))
        print(f"    → '{word}' ({sim:.4f}) {bar}")

In [ ]:
# ── Visualize the gender / royalty offset ─────────────────────────
royalty_words = ['king', 'queen', 'prince', 'princess', 'man', 'woman']
avail = [w for w in royalty_words if w in toy_glove_embeddings]

vecs = np.array([toy_glove_embeddings[w] for w in avail])
pca  = PCA(n_components=2, random_state=42)
xy   = pca.fit_transform(vecs)

color_map = {'king': '#4A90D9', 'prince': '#7BB4E0',
             'queen': '#E87070', 'princess': '#F5A0A0',
             'man': '#7B68EE', 'woman': '#F5A623'}

fig, ax = plt.subplots(figsize=(9, 7))

for i, word in enumerate(avail):
    x, y = xy[i]
    c = color_map.get(word, '#888')
    ax.scatter(x, y, s=250, color=c, zorder=3, alpha=0.9)
    ax.annotate(word, (x, y), fontsize=13, fontweight='bold',
                xytext=(9, 9), textcoords='offset points', color=c)

# Draw offset arrows
pairs = [('king','queen'), ('man','woman'), ('prince','princess')]
for w1, w2 in pairs:
    if w1 in avail and w2 in avail:
        i1, i2 = avail.index(w1), avail.index(w2)
        ax.annotate('', xy=xy[i2], xytext=xy[i1],
                    arrowprops=dict(arrowstyle='->', color='#aaa',
                                   lw=1.5, linestyle='dashed'))

ax.text(0.05, 0.92, 'Dashed arrows show gender offset direction',
        transform=ax.transAxes, fontsize=10, color='#888', style='italic')

ax.set_title('GloVe Vectors in 2D (PCA)\nParallel arrows = consistent relationship direction',
             fontweight='bold', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. 🗺️ Visualizing GloVe Embeddings

Let's use t-SNE to visualize how GloVe organizes words in semantic space.

In [ ]:
# ── t-SNE visualization of GloVe vectors ─────────────────────────
word_groups = {
    'Royalty':    ['king', 'queen', 'prince', 'princess', 'royal', 'kingdom'],
    'People':     ['man', 'woman', 'son', 'daughter', 'family'],
    'Professions':['doctor', 'nurse', 'teacher', 'scientist'],
    'Animals':    ['dog', 'cat', 'mouse'],
    'Geography':  ['france', 'germany', 'paris', 'berlin', 'capital', 'city'],
    'States':     ['ice', 'steam', 'water', 'solid', 'gas', 'hot', 'cold'],
}

group_colors = {
    'Royalty':     '#F5A623',
    'People':      '#7B68EE',
    'Professions': '#4A90D9',
    'Animals':     '#4CAF50',
    'Geography':   '#E87070',
    'States':      '#26A69A',
}

plot_words, plot_groups = [], []
for group, words in word_groups.items():
    for w in words:
        if w in toy_glove_embeddings:
            plot_words.append(w)
            plot_groups.append(group)

plot_vecs = np.array([toy_glove_embeddings[w] for w in plot_words])

if len(plot_vecs) > 10:
    perp = min(10, len(plot_vecs) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perp,
                n_iter=2000, learning_rate='auto', init='random')
    coords = tsne.fit_transform(plot_vecs)
else:
    pca = PCA(n_components=2)
    coords = pca.fit_transform(plot_vecs)

fig, ax = plt.subplots(figsize=(13, 9))
ax.set_facecolor('#f8f9fa')

for i, (word, group) in enumerate(zip(plot_words, plot_groups)):
    x, y = coords[i]
    color = group_colors[group]
    ax.scatter(x, y, s=140, color=color, alpha=0.85, zorder=3)
    ax.annotate(word, (x, y), fontsize=10, fontweight='bold',
                xytext=(5, 5), textcoords='offset points', color=color)

unique_groups = list(dict.fromkeys(plot_groups))
patches = [mpatches.Patch(color=group_colors[g], label=g) for g in unique_groups]
ax.legend(handles=patches, loc='best', fontsize=11, title='Word Groups')

ax.set_title('GloVe Embeddings — t-SNE / PCA Visualization\n(Similar words cluster together)',
             fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('glove_tsne.png', dpi=120, bbox_inches='tight')
plt.show()

print("💡 GloVe clusters are particularly tight because it uses GLOBAL statistics.")
print("   Every word's position is influenced by ALL its co-occurrences in the corpus.")

<a id='10'></a>
## 10. ⚔️ Word2Vec vs GloVe — Head-to-Head Comparison

In [ ]:
# ── Visual comparison diagram ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

comparison = [
    ('Training approach',  'Local window\n(one sentence at a time)',      'Global matrix\n(entire corpus first)'),
    ('Memory usage',       'Low\n(streams through data)',                  'Higher\n(stores V×V matrix)'),
    ('Speed',              'Faster on large corpora',                      'Slower (matrix factorization)'),
    ('Analogy tasks',      'Good',                                         'Excellent (global patterns)'),
    ('Rare words',         'Weaker\n(sees few examples)',                   'Better\n(uses all co-occurrence info)'),
    ('Online learning',    'Yes (add new data)',                           'No (rebuild matrix)'),
    ('Interpretability',   'Black box neural net',                         'Explicit objective\n(fit log counts)'),
    ('Best analogy',       'Taxi driver\n(learns by driving)',             'Librarian\n(indexes everything first)'),
]

categories = [r[0] for r in comparison]
w2v_vals   = [r[1] for r in comparison]
glove_vals = [r[2] for r in comparison]

n = len(categories)
for ax_idx, (ax, vals, title, color) in enumerate([
    (axes[0], w2v_vals,   'Word2Vec', '#4A90D9'),
    (axes[1], glove_vals, 'GloVe',    '#26A69A'),
]):
    ax.set_xlim(0, 1); ax.set_ylim(-0.5, n - 0.5)
    ax.axis('off')
    ax.set_facecolor('white')

    # Title bar
    ax.add_patch(plt.Rectangle((0, n-0.6), 1, 0.7, color=color, alpha=0.9))
    ax.text(0.5, n-0.2, title, ha='center', va='center',
            fontsize=16, fontweight='bold', color='white')

    for i, (cat, val) in enumerate(zip(reversed(categories), reversed(vals))):
        bg = '#f0f4f8' if i % 2 == 0 else 'white'
        ax.add_patch(plt.Rectangle((0, i-0.4), 1, 0.8, color=bg, alpha=0.8))

        # Category label on left panel only
        if ax_idx == 0:
            ax.text(-0.02, i, cat, ha='right', va='center',
                    fontsize=10, color='#444', fontweight='bold',
                    transform=ax.transData)

        ax.text(0.5, i, val, ha='center', va='center',
                fontsize=9.5, color='#333', wrap=True,
                multialignment='center')

plt.suptitle('Word2Vec vs GloVe — Side by Side Comparison', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('word2vec_vs_glove.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final comparison summary ──────────────────────────────────────
print("📊 FINAL COMPARISON SUMMARY")
print("=" * 70)
print()
print("                    WORD2VEC              GLOVE")
print("-" * 70)
rows = [
    ("Approach",    "Prediction task",         "Matrix factorization"),
    ("Data view",   "Local context windows",   "Global co-occurrence counts"),
    ("Speed",       "⚡ Faster",                "🐢 Slower"),
    ("Analogy",     "✅ Good",                  "✅✅ Excellent"),
    ("Rare words",  "Weaker",                   "Better"),
    ("Memory",      "Low",                      "High (V×V matrix)"),
    ("Paper",       "Mikolov et al. 2013",      "Pennington et al. 2014"),
]
for row in rows:
    print(f"  {row[0]:<14} {row[1]:<28} {row[2]}")

print()
print("🎯 PRACTICAL GUIDANCE:")
print("-" * 70)
print("""
  Use Word2Vec when:
  • You have a large, ever-growing corpus (streaming data)
  • Memory is limited
  • You need fast training on huge data

  Use GloVe when:
  • You have a fixed corpus and want the best analogy performance
  • You want globally-consistent embeddings
  • You prefer a mathematically explicit objective function

  Use NEITHER when:
  • You need contextual embeddings ("bank" the river vs "bank" account)
  • Use BERT, RoBERTa, or sentence-transformers instead!
""")

print("✅ In practice, both Word2Vec and GloVe are used as FAST BASELINES")
print("   or in lightweight production systems where BERT is too heavy.")

---

## 🎓 Summary: GloVe Key Points

| Concept | Key Takeaway |
|---|---|
| **Core idea** | Factorize global co-occurrence matrix |
| **Co-occurrence matrix** | Count how often each word pair appears together |
| **Objective** | Dot product of vectors ≈ log(co-occurrence count) |
| **Weighting** | Down-weight very frequent pairs (the, a, ...) |
| **Final vector** | Sum of word vector + context vector |
| **Strength** | Global patterns → better analogy performance |
| **Limitation** | Still static; one vector per word |
| **vs Word2Vec** | Different philosophy, similar quality in practice |

---

### 🔗 What's Next?
- **FastText** — adds character n-gram subword embeddings (handles OOV words)
- **ELMo** — first contextual embeddings (LSTM-based)
- **BERT** — transformer-based contextual embeddings (state of the art 2018)
- **Sentence-BERT** — embeddings for whole sentences